<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/RUL_%EC%98%88%EC%B8%A1_%EB%B0%8F_%EC%8B%9C%EA%B0%81%ED%99%94_%EC%BD%94%EB%93%9C_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch
import torch.nn as nn
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False


def generate_dummy_data(file_path):
    """비선형(가속화) 열화 패턴을 띄는 가상 데이터 생성"""
    print("Generating virtual accelerating degradation data...")
    np.random.seed(42)
    dates = pd.date_range(end=pd.Timestamp.now(), periods=1000, freq='1h')
    mean_values = []

    current_val = 100
    for i in range(1000):
        # 시간이 지날수록 상승폭이 커지는 비선형(2차 함수형) 패턴 + 노이즈
        step_increase = 0.2 + (i / 400)**2 + np.random.normal(0, 0.5)
        current_val += max(0, step_increase + np.random.normal(0, 1.0))
        mean_values.append(current_val)

    df = pd.DataFrame({'start_time': dates, 'mean': mean_values})
    df.to_parquet(file_path)
    print(f"Virtual data saved: {file_path}")


# ----------------------------------------
# PyTorch 비선형 곡선 모델 (Quadratic Curve Fitting)
# 방정식: y = w2 * x^2 + w1 * x + b
# ----------------------------------------
class DegradationCurve(nn.Module):
    def __init__(self):
        super(DegradationCurve, self).__init__()
        # 초기 가중치 설정 (학습을 통해 최적화됨)
        self.w2 = nn.Parameter(torch.tensor(0.001))
        self.w1 = nn.Parameter(torch.tensor(0.1))
        self.b = nn.Parameter(torch.tensor(100.0))

    def forward(self, x):
        return self.w2 * (x ** 2) + self.w1 * x + self.b


def train_and_predict_curve(df, threshold=1100):
    """PyTorch GPU를 이용해 최적의 열화 곡선 파라미터를 찾고 도달 시점 계산"""
    print("\n1. Preparing data for Curve Fitting...")

    # 시간 단위 스케일링: 타임스탬프 수치가 너무 크면 GPU 학습 시 기울기 폭발(NaN)이 발생함
    # 따라서 시작 시간을 0으로 맞추고 '경과 시간(Hours)'을 X값으로 사용
    start_time_val = df['start_time'].iloc[0]
    df['hours_passed'] = (df['start_time'] - start_time_val).dt.total_seconds() / 3600.0

    X = torch.FloatTensor(df['hours_passed'].values)
    y = torch.FloatTensor(df['mean'].values)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"2. Optimizing Curve Parameters on device: {device} 🚀")

    X, y = X.to(device), y.to(device)
    model = DegradationCurve().to(device)

    # 손실함수 및 옵티마이저 (LBFGS 또는 Adam 사용)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

    # GPU를 활용한 곡선 최적화 학습
    epochs = 2000
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        if (epoch+1) % 500 == 0:
            print(f" - Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.2f}")

    # 학습된 파라미터 추출
    w2 = model.w2.item()
    w1 = model.w1.item()
    b = model.b.item()

    print(f"\n3. Learned Equation: y = {w2:.5f}*x^2 + {w1:.5f}*x + {b:.2f}")

    # [수학적 역산] 1100에 도달하는 정확한 x(시간) 구하기
    # 방정식: w2*x^2 + w1*x + (b - threshold) = 0
    # 근의 공식 적용: x = (-w1 + sqrt(w1^2 - 4*w2*(b-threshold))) / (2*w2)

    discriminant = w1**2 - 4 * w2 * (b - threshold)

    if discriminant < 0 or w2 <= 0:
        print("Warning: The curve does not strictly accelerate towards the threshold.")
        # 2차항이 유효하지 않으면 1차(선형)으로 근사하여 계산
        hours_to_target = (threshold - b) / w1
    else:
        # 양의 근(미래 시간)을 선택
        root1 = (-w1 + np.sqrt(discriminant)) / (2 * w2)
        root2 = (-w1 - np.sqrt(discriminant)) / (2 * w2)
        hours_to_target = max(root1, root2)

    # 도달 예상 시간 계산
    predicted_time = start_time_val + pd.Timedelta(hours=hours_to_target)

    print("\n==================================================")
    print(f" 🎯 비선형 곡선 1100 도달 예상 시점 : {predicted_time.strftime('%Y년 %m월 %d일 %H시 %M분 %S초')}")
    print("==================================================\n")

    # 시각화를 위한 궤적 데이터 생성
    future_hours = np.linspace(0, hours_to_target * 1.1, 500) # 도달 시점 이후 약간 더 연장해서 선 그리기
    future_X = torch.FloatTensor(future_hours).to(device)

    with torch.no_grad():
        future_y = model(future_X).cpu().numpy()

    future_dates = [start_time_val + pd.Timedelta(hours=h) for h in future_hours]

    return future_dates, future_y, predicted_time, model


def visualize_curve_prediction(df, future_dates, future_y, predicted_time, threshold=1100):
    """최적화된 비선형 곡선 예측 결과 시각화"""
    print("4. Generating Visualization Chart...")
    plt.figure(figsize=(12, 7))

    # 1. 실제 과거 데이터 (노이즈 포함)
    plt.plot(df['start_time'], df['mean'], color='gray', alpha=0.5, linewidth=1.5, label='Actual Sensor Data (Noise)')

    # 2. PyTorch 최적화 예측 곡선 (2차 함수 궤적)
    plt.plot(future_dates, future_y, color='#ff7f0e', linestyle='-', linewidth=3,
             label='PyTorch Non-linear Curve Fit')

    # 예측 적중 시점 마킹
    plt.scatter([predicted_time], [threshold], color='#d62728', s=120, zorder=5)

    # 임계치 라인
    plt.axhline(y=threshold, color='red', linestyle=':', linewidth=2, label=f'Threshold ({threshold})')

    # 텍스트 주석
    plt.annotate(f'Predicted Hit:\n{predicted_time.strftime("%Y-%m-%d %H:%M")}',
                 xy=(predicted_time, threshold), xytext=(-80, 30), textcoords='offset points',
                 arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=.2", color='#d62728'),
                 fontsize=12, fontweight='bold', color='#d62728')

    plt.title('Equipment RUL Prediction via PyTorch Curve Optimization', fontsize=16, fontweight='bold')
    plt.xlabel('Date / Time', fontsize=12)
    plt.ylabel('Sensor Value (mean)', fontsize=12)

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=45)

    # Y축 범위 깔끔하게
    plt.ylim(min(df['mean']) - 50, threshold + 100)

    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend(fontsize=12, loc='upper left')
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    parquet_file = 'sample_sensor_data_curve.parquet'

    # 가상의 가속화 패턴 데이터 생성
    if not os.path.exists(parquet_file):
        generate_dummy_data(parquet_file)

    try:
        df = pd.read_parquet(parquet_file)
        df['start_time'] = pd.to_datetime(df['start_time'])
        df = df.sort_values('start_time').reset_index(drop=True)

        # PyTorch 기반 곡선 최적화 및 시점 예측
        future_dates, future_y, pred_time, model = train_and_predict_curve(df, threshold=1100)

        # 차트 출력
        visualize_curve_prediction(df, future_dates, future_y, pred_time)

    except Exception as e:
        print(f"Error occurred: {e}")